In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

from src.portfolio.positions import Portfolio
from src.stress.historical import stress_window
from src.stress.hypothetical import apply_vol_scaling
from src.stress.scenarios import increase_correlation
from src.stress.reporting import stress_loss_distribution
from src.stress.run_stress import (
    stress_var,
    run_stress)
from src.utils.config import load_config
from src.data.loaders import load_prices
from src.data.features import compute_log_returns
from src.portfolio.pnl import compute_portfolio_pnl

### Stress test historical crisis (Covid period)

In [6]:
# Load data and compute returns
portfolio_cfg = load_config(Path("../configs/portfolio.yaml"))
risk_cfg = load_config(Path("../configs/risk.yaml"))
stress_cfg = load_config(Path("../configs/stress.yaml"))

tickers = portfolio_cfg["portfolio"]["tickers"]
notional = portfolio_cfg["portfolio"]["notional"]

prices = load_prices(
    tickers,
    start=risk_cfg["risk"]["start_date"],
    end=risk_cfg["risk"]["end_date"]
)

returns = compute_log_returns(prices)

portfolio = Portfolio.from_equal_weight(
    tickers=tickers,
    notional=notional
)

stress_start = stress_cfg["stress"]["scenarios"]["covid_crash"]["start"]
stress_end = stress_cfg["stress"]["scenarios"]["covid_crash"]["end"]
covid_returns = stress_window(
    returns,
    stress_start,
    stress_end
)

covid_losses = stress_loss_distribution(
    covid_returns,
    portfolio.weights,
    portfolio.notional
)

covid_tail = covid_losses.quantile(0.99)
print(f"COVID-19 Stress Test Historical VaR (99%): {covid_tail:.2f}")

Cached prices available
Cached data covers request. Loading from cache.
COVID-19 Stress Test Historical VaR (99%): 1202261.33


In [7]:
covid_var = run_stress(
    returns,
    portfolio,
    start=stress_start,
    end=stress_end,
    model="parametric_ewma",
    alpha=stress_cfg["stress"]["alpha"] # = 0.99
)
normal_var = stress_var(
    returns,
    portfolio,
    model="parametric_ewma",
    alpha=stress_cfg["stress"]["alpha"]
)